# Modelado Semántico Aplicado: Ontologías, RDF/OWL, SKOS y Knowledge Graphs

## Objetivos
- Construir una ontología mínima con **RDF/OWL** en Python (rdflib).
- Consultar con **SPARQL** para responder preguntas de negocio.
- Modelar una **taxonomía con SKOS** y relacionarla con datos.
- Mapear contratos de datos y columnas a **IRIs semánticos**.
- Sentar bases para **knowledge graphs** y semántica empresarial.

## 1. Introducción Rápida a Estándares
- **RDF**: grafo de tripletas (sujeto, predicado, objeto).
- **RDFS/OWL**: vocabularios para clases, propiedades y restricciones.
- **SKOS**: estándar para taxonomías/tesauros (broader/narrower/related).
- **SPARQL**: lenguaje de consulta sobre grafos RDF.
- **IRI**: identificador global (ej: `https://example.com/ontology/Customer`).

## 2. Ontología mínima con rdflib
Definimos clases: `Customer`, `Order`, `Product`. Propiedades: `hasOrder`, `hasProduct`, `totalAmount`.

In [ ]:
# Si no tienes rdflib: pip install rdflib  (ejecutar en tu entorno)
from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, XSD, URIRef

EX = Namespace('https://example.com/ontology/')
g = Graph()
g.bind('ex', EX)
g.bind('owl', OWL)

# Definición de clases
for cls in ['Customer','Order','Product']:
    g.add((EX[cls], RDF.type, OWL.Class))

# Propiedades
g.add((EX.hasOrder, RDF.type, OWL.ObjectProperty))
g.add((EX.hasProduct, RDF.type, OWL.ObjectProperty))
g.add((EX.totalAmount, RDF.type, OWL.DatatypeProperty))

# Dominio y rango
g.add((EX.hasOrder, RDFS.domain, EX.Customer))
g.add((EX.hasOrder, RDFS.range, EX.Order))
g.add((EX.hasProduct, RDFS.domain, EX.Order))
g.add((EX.hasProduct, RDFS.range, EX.Product))
g.add((EX.totalAmount, RDFS.domain, EX.Order))
g.add((EX.totalAmount, RDFS.range, XSD.decimal))

# Instancias y relaciones
cust1 = EX['customer/1001']
ord1 = EX['order/5001']
prod1 = EX['product/2007']
g.add((cust1, RDF.type, EX.Customer))
g.add((ord1, RDF.type, EX.Order))
g.add((prod1, RDF.type, EX.Product))
g.add((cust1, EX.hasOrder, ord1))
g.add((ord1, EX.hasProduct, prod1))
g.add((ord1, EX.totalAmount, Literal('1200.00', datatype=XSD.decimal)))

print(g.serialize(format='turtle').decode())

## 3. Consultas SPARQL
Ejemplo: obtener pedidos y montos por cliente.

In [ ]:
q = """
PREFIX ex: <https://example.com/ontology/>
SELECT ?customer ?order ?amount
WHERE {
  ?customer a ex:Customer ; ex:hasOrder ?order .
  ?order ex:totalAmount ?amount .
}
"""
for row in g.query(q):
    print(row)

## 4. Taxonomías con SKOS
Creamos una taxonomía de producto y usamos `skos:broader`/`skos:narrower`.

In [ ]:
from rdflib.namespace import SKOS

g.bind('skos', SKOS)
electronics = EX['concept/electronics']
phones = EX['concept/electronics/phones']
smartphones = EX['concept/electronics/phones/smartphones']

for c,label in [(electronics,'Electronics'),(phones,'Phones'),(smartphones,'Smartphones')]:
    g.add((c, RDF.type, SKOS.Concept))
    g.add((c, SKOS.prefLabel, Literal(label)))

g.add((phones, SKOS.broader, electronics))
g.add((smartphones, SKOS.broader, phones))

# Consulta SKOS
q = """
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?narrower ?label
WHERE { ?narrower skos:broader <https://example.com/ontology/concept/electronics> ; skos:prefLabel ?label . }
"""
for row in g.query(q):
    print(row)

## 5. Mapear Contratos de Datos a Ontologías
Vinculamos columnas a IRIs (semántica explícita) para interoperabilidad.

In [ ]:
# Contrato de datos simple
contract = {
  'dataset': 'orders_silver',
  'columns': {
    'customer_id': 'https://example.com/ontology/Customer',
    'order_id': 'https://example.com/ontology/Order',
    'total_amount': 'https://example.com/ontology/totalAmount'
  }
}

def validar_contrato(contract, graph):
    missing = []
    for col, iri in contract['columns'].items():
        if (URIRef(iri), None, None) not in graph and (None, None, URIRef(iri)) not in graph:
            missing.append((col, iri))
    return missing

print('Faltantes (si no están definidos en la ontología):')
print(validar_contrato(contract, g))

## 6. Hacia Knowledge Graphs
- Un **KG** conecta ontologías + datos operativos + metadatos.
- Casos: 360 de cliente, recomendaciones, cumplimiento (linaje semántico).
- Integración con catálogo y lineage aporta descubrimiento y trazabilidad.

## 7. Ejercicio
1) Añade clase `Channel` y relación `purchasedVia` a la ontología.
2) Crea 2 instancias de `Order` con distintos canales.
3) Escribe una consulta SPARQL para contar pedidos por canal.